In [1]:
import pandas as pd
df = pd.read_csv(r'C:\Users\choiy\Desktop\aegis_prac\data\training\p000001.psv', sep='|')
df

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0
5,110.0,91.0,NaN,NaN,NaN,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,6,0
6,108.0,92.0,36.11,123.0,77.00,NaN,29.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,7,0
7,106.0,90.5,NaN,93.0,76.33,NaN,29.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,8,0
8,104.0,95.0,NaN,133.0,88.33,NaN,26.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,9,0
9,102.0,91.0,NaN,134.0,87.33,NaN,30.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,10,0


In [3]:
print(df.shape)
print(df.columns.tolist())
df.head()

(54, 41)
['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel']


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0


In [ ]:
import os
files = os.listdir(r'C:\Users\choiy\Desktop\aegis_prac\data\training')
print(f"전체 환자 수: {len(files)}")

전체 환자 수: 20336


In [6]:
import glob
all_files = glob.glob(r"C:\Users\choiy\Desktop\aegis_prac\data\training\*.psv")

sepesis_count = 0
for f in all_files:
    df_temp = pd.read_csv(f, sep='|')
    if df_temp['SepsisLabel'].max() == 1:
        sepesis_count += 1

print(f"패혈증 환자 수 : {sepesis_count}")
print(f"정상 환자 수 : {len(all_files) - sepesis_count}")
print(f"패혈증 비율 : {sepesis_count/len(all_files)*100:.1f}%")

패혈증 환자 수 : 1790
정상 환자 수 : 18546
패혈증 비율 : 8.8%


In [12]:
import sys

sys.path.append(r'C:\Users\choiy\Desktop\aegis_prac')

from config import *
import duckdb

with duckdb.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS patients AS
        SELECT * FROM read_csv_auto(
            'data/training/*.psv',
            delim='|',
            header=true,
            filename=true
        )
    """)
    print(conn.execute("SELECT COUNT(*) FROM patients").fetchone()[0])

790215


In [24]:
import pandas as pd

conn = duckdb.connect(DB_PATH)

pd.set_option('display.max_columns', None)  # 모든 컬럼 표시
pd.set_option('display.max_rows', None)     # 모든 행 표시
pd.set_option('display.width', None)        # 너비 제한 없음

In [ ]:
df = conn.execute("SELECT HR, Temp, Lactate FROM patients LIMIT 10").df()
print(df)
print(df.dtypes)

      HR   Temp  Lactate
0    NaN    NaN      NaN
1   97.0    NaN      NaN
2   89.0    NaN      NaN
3   90.0    NaN      NaN
4  103.0    NaN      NaN
5  110.0    NaN      NaN
6  108.0  36.11      NaN
7  106.0    NaN      NaN
8  104.0    NaN      NaN
9  102.0    NaN      NaN
HR         float64
Temp       float64
Lactate    float64
dtype: object


In [ ]:
columns = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp',
            'BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2','AST',
            'BUN','Alkalinephos','Calcium','Chloride','Creatinine',
            'Bilirubin_direct','Glucose','Lactate','Magnesium',
            'Phosphate','Potassium','Bilirubin_total','TroponinI',
            'Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']

query = "SELECT " + ",\n".join([
    f"ROUND(100.0 * SUM(CASE WHEN {c} IS NULL OR isnan({c}) THEN 1 ELSE 0 END) / COUNT(*), 1) AS {c}"
    for c in columns
]) + " FROM patients"

df = conn.execute(query).df()
print(df.T.sort_values(0))  # .T하면 컬럼이 0이 됨.

                     0
HR                 7.7
Resp               9.8
MAP               10.2
O2Sat             12.0
SBP               15.2
DBP               48.1
Temp              66.2
FiO2              85.8
Glucose           87.8
Hct               88.2
pH                88.5
Potassium         89.1
BaseExcess        89.6
Hgb               91.2
PaCO2             91.2
Chloride          91.7
BUN               91.8
HCO3              91.9
Magnesium         92.2
WBC               92.5
Creatinine        93.4
Platelets         93.5
SaO2              95.0
Calcium           95.0
Phosphate         95.0
PTT               95.2
Lactate           96.6
Alkalinephos      98.5
AST               98.5
Bilirubin_total   98.8
Fibrinogen        99.2
Bilirubin_direct  99.9
TroponinI         99.9


In [ ]:
# 모델링할 때 그냥 안 쓰기
# 원본 데이터는 건드리지 않고 제외만 하기
# Hgb, PaCO2, Chloride, BUN, HCO3, Magnesium, WBC, Creatinine, Platelets, SaO2              
# Calcium           
# ...
# Bilirubin_total   
# Fibrinogen       
# Bilirubin_direct  
# TroponinI        

